# Semi-supervised clustering (SSC) baseline

Bair & Tibshirani (2004), the clustering baseline used in the VaDeSC paper:

1. score every feature with a univariate Cox model,
2. keep the most survival-related features,
3. run k-means on those features only.

So survival information is used only to choose the features, while VaDeSC uses it inside the model. Implementation in `src/ssc.py` (checked against the official VaDeSC baseline code, `baselines/ssc/sscBair.py`).

The number of features has to be chosen. Two ways:
- **val log-rank**: the number that gives the most separated survival curves (largest log-rank statistic) on the validation set. Uses only survival data, so it works on real datasets too.
- **oracle**: the number that gives the best agreement (NMI) with the *true* clusters on the training set. This is what the official VaDeSC code does for the simulated data (the official code notes that this is optimistic). Only possible on simulated data.

The official code ranks features by hazard ratio (largest first), which only keeps features where larger values mean higher risk. Bair & Tibshirani rank by the absolute Cox score, which keeps both directions. We use the absolute score by default and also report the official ranking.

In [1]:
import sys
sys.path.append("..")
from dataclasses import replace

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from src.data import load_simulated, load_sksurv, load_support
from src.metrics import logrank_chi2
from src.ssc import fit_ssc
from src.vadesc import evaluate_vadesc, train_vadesc

QUICK = False
SEEDS = [0, 1] if QUICK else [0, 1, 2, 3, 4]
N, P, K, D_LAT = (2000, 50, 3, 8) if QUICK else (10_000, 1000, 3, 16)  # same as 03_simulated_benchmark_multi
REAL_SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = 60 if QUICK else 300

## 1. Simulated data

Same data as `03_simulated_benchmark_multi` (10,000 patients, 1,000 features, 3 clusters, seeds 0-4), so the VaDeSC and VaDE numbers can be taken from `results/sim_by_seed.csv`. Clusters are evaluated on the test set.

In [2]:
def cluster_scores(d, clusters):
    return {"ARI": adjusted_rand_score(d.clusters_true_te, clusters),
            "NMI": normalized_mutual_info_score(d.clusters_true_te, clusters),
            "log-rank chi2": logrank_chi2(d, clusters)}

rows = []
for s in SEEDS:
    d = load_simulated(n=N, p=P, k=K, d_lat=D_LAT, seed=s)
    km = KMeans(K, n_init=10, random_state=s).fit(d.X_tr)
    rows.append({"seed": s, "model": "k-means (all features)", "n features": P, **cluster_scores(d, km.predict(d.X_te))})
    for name, how, rank in [("SSC (val log-rank)", "val logrank", "score"),
                            ("SSC (oracle)", "oracle", "score"),
                            ("SSC (oracle, official ranking)", "oracle", "hazard_ratio")]:
        m = fit_ssc(d, K, how=how, rank=rank, seed=s, c_tr=d.clusters_true_tr)
        rows.append({"seed": s, "model": name, "n features": m.n_features, **cluster_scores(d, m.predict(d.X_te))})
    print("seed", s, "done", flush=True)
sim = pd.DataFrame(rows)
sim.pivot_table(index="model", columns="seed", values="n features")

seed 0 done


seed 1 done


seed 2 done


seed 3 done


seed 4 done


seed,0,1,2,3,4
model,,,,,
SSC (oracle),551.0,301.0,901.0,901.0,901.0
"SSC (oracle, official ranking)",301.0,151.0,901.0,751.0,301.0
SSC (val log-rank),20.0,20.0,100.0,50.0,10.0
k-means (all features),1000.0,1000.0,1000.0,1000.0,1000.0


In [3]:
order = ["VaDeSC", "VaDE", "k-means (all features)", "SSC (val log-rank)", "SSC (oracle)", "SSC (oracle, official ranking)"]
prev = pd.read_csv("../results/sim_by_seed.csv")
prev = prev[prev.model.isin(["VaDeSC", "VaDE"]) & prev.seed.isin(SEEDS)]
if QUICK:  # the saved results are for the full-size data
    prev = prev.iloc[:0]
both = pd.concat([prev[["seed", "model", "ARI", "log-rank chi2"]], sim], ignore_index=True)
both.groupby("model")[["ARI", "NMI", "log-rank chi2"]].agg(["mean", "std"]).reindex(order).round(3)

ARI           NMI        log-rank chi2  \
                                 mean    std   mean    std          mean   
model                                                                      
VaDeSC                          0.323  0.105    NaN    NaN        24.291   
VaDE                            0.125  0.074    NaN    NaN        16.178   
k-means (all features)          0.037  0.023  0.046  0.030        35.623   
SSC (val log-rank)              0.018  0.013  0.021  0.014       306.239   
SSC (oracle)                    0.046  0.025  0.052  0.030        95.318   
SSC (oracle, official ranking)  0.053  0.024  0.056  0.026        88.391   

                                        
                                   std  
model                                   
VaDeSC                          22.967  
VaDE                             7.478  
k-means (all features)          17.485  
SSC (val log-rank)              50.725  
SSC (oracle)                    68.756  
SSC (oracle, official ranking)  79.877

VaDeSC and VaDE rows come from `03_simulated_benchmark_multi` (same seeds, same data; NMI was not saved there).

## 2. Real datasets

No true clusters here, so we compare how different the survival of the clusters is on the test set (log-rank statistic, higher = more separated), for k-means on all features, SSC and VaDeSC. The log-rank statistic grows with the number of patients, so compare methods within a dataset, not across datasets.

- SUPPORT: baseline-only features (no leakage), 4 clusters, VaDeSC config from the paper.
- FLChain, GBSG2, WHAS500: 2 clusters, the paper's FLChain config for VaDeSC. WHAS500's `los` is dropped (equals the survival time for patients who died in hospital).

Seeds change the data split and the model initialisation.

In [4]:
def drop_features(d, names):
    keep = [i for i, f in enumerate(d.features) if f not in names]
    return replace(d, X_tr=d.X_tr[:, keep], X_va=d.X_va[:, keep], X_te=d.X_te[:, keep],
                   features=[d.features[i] for i in keep])

DATASETS = {
    "SUPPORT": (lambda s: load_support(drop_post_baseline=True, seed=s), dict(K=4, d_lat=16, weibull_k=2.0)),
    "FLChain": (lambda s: load_sksurv("FLChain", seed=s), dict(K=2, d_lat=4, weibull_k=1.0)),
    "GBSG2": (lambda s: load_sksurv("GBSG2", seed=s), dict(K=2, d_lat=4, weibull_k=1.0)),
    "WHAS500": (lambda s: drop_features(load_sksurv("WHAS500", seed=s), ["los"]), dict(K=2, d_lat=4, weibull_k=1.0)),
}

def summary(d, clusters):
    return {"log-rank chi2": logrank_chi2(d, clusters), "smallest cluster": np.bincount(clusters).min() / len(clusters)}

rows, picked = [], {}
for name, (load, cfg) in DATASETS.items():
    for s in REAL_SEEDS:
        d = load(s)
        km = KMeans(cfg["K"], n_init=10, random_state=s).fit(d.X_tr)
        rows.append({"dataset": name, "seed": s, "model": "k-means (all features)", **summary(d, km.predict(d.X_te))})
        ssc = fit_ssc(d, cfg["K"], how="val logrank", seed=s)
        rows.append({"dataset": name, "seed": s, "model": "SSC (val log-rank)", "n features": ssc.n_features,
                     **summary(d, ssc.predict(d.X_te))})
        picked[(name, s)] = [d.features[i] for i in ssc.features]
        vad = train_vadesc(d, epochs=EPOCHS, seed=s, **cfg)
        m, cl, _ = evaluate_vadesc(vad, d)
        rows.append({"dataset": name, "seed": s, "model": "VaDeSC", "C-index": m["C-index"], **summary(d, cl)})
        print(name, "seed", s, "done", flush=True)
real = pd.DataFrame(rows)
(real.groupby(["dataset", "model"])[["log-rank chi2", "smallest cluster"]].agg(["mean", "std"])
     .reindex(list(DATASETS), level=0).round(2))

SUPPORT seed 0 done


SUPPORT seed 1 done


SUPPORT seed 2 done


FLChain seed 0 done


FLChain seed 1 done


FLChain seed 2 done


... (3 outputs left out)

WHAS500 seed 0 done


WHAS500 seed 1 done


WHAS500 seed 2 done


log-rank chi2         smallest cluster      
                                        mean     std             mean   std
dataset model                                                              
SUPPORT SSC (val log-rank)            257.50   89.65             0.08  0.06
        VaDeSC                        254.85  103.60             0.03  0.00
        k-means (all features)         41.28    7.65             0.03  0.00
FLChain SSC (val log-rank)            341.06  101.49             0.19  0.16
        VaDeSC                        233.31  137.66             0.05  0.03
        k-means (all features)        310.14   85.72             0.13  0.06
GBSG2   SSC (val log-rank)              2.54    0.94             0.17  0.03
        VaDeSC                          1.13    0.37             0.32  0.24
        k-means (all features)          0.90    0.79             0.45  0.03
WHAS500 SSC (val log-rank)             23.95   12.49             0.44  0.06
        VaDeSC                         14.10    9.23             0.37  0.09
        k-means (all features)         20.51    3.49             0.45  0.03

In [5]:
# number of features SSC chose, and which ones (seed 0)
print(real[real.model == "SSC (val log-rank)"].pivot_table(index="dataset", columns="seed", values="n features"))
for name in DATASETS:
    print(f"{name}: {picked[(name, REAL_SEEDS[0])][:10]}")

seed       0     1    2
dataset                
FLChain  5.0   2.0  5.0
GBSG2    1.0   1.0  1.0
SUPPORT  2.0   5.0  2.0
WHAS500  1.0  10.0  5.0
SUPPORT: ['scoma', 'avtisst']
FLChain: ['age', 'kappa', 'lambda', 'flc.grp_10', 'creatinine']
GBSG2: ['pnodes']
WHAS500: ['age']


## Notes

- SSC has no risk score, so it has no C-index (the official code writes a constant 0.5). We compare clusters only.
- The univariate Cox fits are done with the score test at $\beta = 0$ (closed form, all features at once), as in Bair & Tibshirani. On WHAS500 its ranking agrees with fitting a separate lifelines Cox model per feature (rank correlation 0.99).
- A large log-rank statistic only says that the clusters have different survival, not that they are meaningful groups; a tiny cluster of very sick patients can give a large value. The smallest-cluster column is there to spot that.